<a href="https://colab.research.google.com/github/Ayseatci/DI725_Assignment1/blob/dev/notebooks/preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://github.com/Ayseatci/DI725_Assignment1

In [14]:
import re
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from google.colab import userdata

!git clone -b dev https://github.com/Ayseatci/DI725_Assignment1.git

%cd DI725_Assignment1

Cloning into 'DI725_Assignment1'...
remote: Enumerating objects: 115, done.
remote: Counting objects: 100% (115/115), done.
remote: Compressing objects: 100% (88/88), done.
remote: Total 115 (delta 59), reused 61 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (115/115), 1.03 MiB | 3.32 MiB/s, done.
Resolving deltas: 100% (59/59), done.
/content/DI725_Assignment1/DI725_Assignment1/DI725_Assignment1


## Preprocessing train data

In [15]:
# Loading train data
df = pd.read_csv("data/raw/train.csv")
removed_rows = df[~df["conversation"].str.contains(r'(?i)customer\s*:', na=False)]
print(f"Sample of removed conversations:\n{removed_rows['conversation'].head()}")

df = df[df["conversation"].str.contains(r'(?i)customer\s*:', na=False)].copy()


# Dropping NaNs
categorical_cols = ["issue_area", "issue_category", "product_category", "agent_experience_level"]
df = df.dropna(subset=["conversation", "customer_sentiment"] + categorical_cols).copy()

# Text preprocessing and replacing tags for customer and agent
def preprocess_dialogue(text):
    text = str(text).strip()
    text = re.sub(r"\r\n|\r", "\n", text)
    text = re.sub(r"(?i)\bcustomer\s*:", "<cust> ", text)
    text = re.sub(r"(?i)\bagent\s*:", "<agent> ", text)
    text = re.sub(r"[ \t]+", " ", text)
    return re.sub(r"\n+", "\n", text).strip()

df["text"] = df["conversation"].apply(preprocess_dialogue)

sentiment_map = {"negative": 0, "neutral": 1, "positive": 2}

df["labels"] = df["customer_sentiment"].map(sentiment_map)

Sample of removed conversations:
191    Agent: You're welcome, Jane. Have a great day!
286    Agent: You're welcome, Jane. Have a great day!
750    Agent: You're welcome, Jane. Have a great day!
Name: conversation, dtype: object


The initial preprocessing step implements a customer presence filter, which detects dialogues that do not consist of a customer line in the conversation. These are removed from both the training and testing sets. This step prevents the neutral bias and helps the model to interpret emotion rather than its ability to handle noise.

A normalization process to prepare the text for the Transformer's self-attention mechanism is applied to filtered conversation. The dialogue is standardized by replacing "Customer:" and "Agent:" identifiers with special tokens <cust> and <agent> to provide structural boundaries to the model. This allows the RoBERTa layers to distinguish between the source of the sentiment and the conetext of the interaction. Whitespace and line breaks is collapsed so that the input sequence is further optimized. This ensures that the model allocates its limited positional embeddings to meaningful vocabulary.


Furthermore, sentiment categories are mapped to ids so they can be utilized by the model.



In [16]:
df.columns

Index(['issue_area', 'issue_category', 'issue_sub_category',
       'issue_category_sub_category', 'customer_sentiment', 'product_category',
       'product_sub_category', 'issue_complexity', 'agent_experience_level',
       'agent_experience_level_desc', 'conversation', 'text', 'labels'],
      dtype='object')

In [17]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["labels"],
    random_state=42
)

train_path = "data/preprocessed/preprocessed_train.csv"
val_path = "data/preprocessed/preprocessed_val.csv"

train_df.to_csv(train_path, index=False)
val_df.to_csv(val_path, index=False)

# Auth and Setup
token = userdata.get('GITHUB_TOKEN')
!git config --global user.email "ayseatci00@gmail.com"
!git config --global user.name "Ayseatci"
!git remote set-url origin https://{token}@github.com/Ayseatci/DI725_Assignment1.git

!git add {train_path} {val_path}
!git commit -m "Add stratified preprocessed train and validation datasets"
!git push origin dev

[dev d2078b3] Add stratified preprocessed train and validation datasets
 2 files changed, 2 insertions(+), 2 deletions(-)
Enumerating objects: 11, done.
Counting objects: 100% (11/11), done.
Delta compression using up to 2 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (6/6), 782 bytes | 782.00 KiB/s, done.
Total 6 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/Ayseatci/DI725_Assignment1.git
   8d20f36..d2078b3  dev -> dev


A stratified splitting is done to create the validation and train datasets, these are then pushed to the preprocessed file in the github repo.

In [18]:
val_df[val_df["customer_sentiment"] == "positive"]

,issue_area,issue_category,issue_sub_category,issue_category_sub_category,customer_sentiment,product_category,product_sub_category,issue_complexity,agent_experience_level,agent_experience_level_desc,conversation,text,labels
958,Order,Product Information and Tags,Finding seller's returns policy,Product Information and Tags -> Finding seller...,positive,Appliances,Kitchen Chimney,medium,experienced,"confidently handles complex customer issues, e...","Customer: Hi, I'm calling because I am interes...","<cust> Hi, I'm calling because I am interested...",2
182,Order,Product Information and Tags,Finding seller's returns policy,Product Information and Tags -> Finding seller...,positive,Men/Women/Kids,T-Shirt,less,junior,"handles customer inquiries independently, poss...","Agent: Hello, thank you for calling BrownBox c...","<agent> Hello, thank you for calling BrownBox ...",2
250,Order,Product Information and Tags,Finding seller's returns policy,Product Information and Tags -> Finding seller...,positive,Men/Women/Kids,Sweatshirt,medium,junior,"handles customer inquiries independently, poss...",Agent: Thank you for calling BrownBox Customer...,<agent> Thank you for calling BrownBox Custome...,2


In [19]:
removed_rows

,issue_area,issue_category,issue_sub_category,issue_category_sub_category,customer_sentiment,product_category,product_sub_category,issue_complexity,agent_experience_level,agent_experience_level_desc,conversation
191,Order,Order Delivery Issues,Package shows as delivered but cannot be found,Order Delivery Issues -> Package shows as deli...,negative,Men/Women/Kids,Shorts,less,junior,"handles customer inquiries independently, poss...","Agent: You're welcome, Jane. Have a great day!"
286,Order,Order Delivery Issues,Package shows as delivered but cannot be found,Order Delivery Issues -> Package shows as deli...,negative,Men/Women/Kids,T-Shirt,less,inexperienced,"may struggle with ambiguous queries, rely on c...","Agent: You're welcome, Jane. Have a great day!"
750,Order,Order Delivery Issues,Order approved but not shipped,Order Delivery Issues -> Order approved but no...,negative,Appliances,Sandwich Maker,less,experienced,"confidently handles complex customer issues, e...","Agent: You're welcome, Jane. Have a great day!"


The removed rows that do not include customer lines are displayed.

## Preprocessing test data

In [20]:
# Loading test data
df = pd.read_csv("data/raw/test.csv")

# Dropping NaNs
categorical_cols = ["issue_area", "issue_category", "product_category", "agent_experience_level"]
df = df.dropna(subset=["conversation", "customer_sentiment"] + categorical_cols).copy()


# Text preprocessing and replacing tags for customer and agent
def preprocess_dialogue(text):
    text = str(text).strip()
    text = re.sub(r"\r\n|\r", "\n", text)
    text = re.sub(r"(?i)\bcustomer\s*:", "<cust> ", text)
    text = re.sub(r"(?i)\bagent\s*:", "<agent> ", text)
    text = re.sub(r"[ \t]+", " ", text)
    return re.sub(r"\n+", "\n", text).strip()

df["text"] = df["conversation"].apply(preprocess_dialogue)

df["labels"] = df["customer_sentiment"].map(sentiment_map)

Sample of removed conversations:
Series([], Name: conversation, dtype: object)


The 'no customer line' filtering is not done to the test data.

In [21]:
df.count()

,0
issue_area,30
issue_category,30
issue_sub_category,30
issue_category_sub_category,30
customer_sentiment,30
product_category,30
product_sub_category,30
issue_complexity,30
agent_experience_level,30
agent_experience_level_desc,30


In [22]:
df.head()

,issue_area,issue_category,issue_sub_category,issue_category_sub_category,customer_sentiment,product_category,product_sub_category,issue_complexity,agent_experience_level,agent_experience_level_desc,conversation,text,labels
0,Shopping,Pricing and Discounts,Discounts through exchange offers,Pricing and Discounts -> Discounts through exc...,negative,Appliances,Hand Blender,less,experienced,"confidently handles complex customer issues, e...",Agent: Thank you for calling BrownBox Customer...,<agent> Thank you for calling BrownBox Custome...,0
1,Login and Account,Account Reactivation and Deactivation,Reactivating an inactive account,Account Reactivation and Deactivation -> React...,negative,Men/Women/Kids,Wrist Watch,medium,junior,"handles customer inquiries independently, poss...",Agent: Thank you for calling BrownBox Customer...,<agent> Thank you for calling BrownBox Custome...,0
2,Cancellations and returns,Cash on Delivery (CoD) Refunds,Refund timelines for Cash on Delivery returns,Cash on Delivery (CoD) Refunds -> Refund timel...,negative,Appliances,Induction Cooktop,less,junior,"handles customer inquiries independently, poss...",Agent: Thank you for calling BrownBox Customer...,<agent> Thank you for calling BrownBox Custome...,0
3,Order,Order Delivery Issues,Package shows as delivered but cannot be found,Order Delivery Issues -> Package shows as deli...,negative,Men/Women/Kids,Sunglas,high,junior,"handles customer inquiries independently, poss...",Agent: Thank you for calling BrownBox Customer...,<agent> Thank you for calling BrownBox Custome...,0
4,Cancellations and returns,Pickup and Shipping,Reimbursement of courier charges for return,Pickup and Shipping -> Reimbursement of courie...,negative,Electronics,Computer Monitor,medium,experienced,"confidently handles complex customer issues, e...",Agent: Thank you for calling BrownBox Customer...,<agent> Thank you for calling BrownBox Custome...,0


The  preprocessing steps applied to training data are done on the test dataset.

In [23]:
# Pushing preprocessed data under preprocessed file in GitHub repository
file_path = "data/preprocessed/preprocessed_test.csv"
df.to_csv(file_path, index=False)

# Auth and Push
token = userdata.get('GITHUB_TOKEN')
!git config --global user.email "ayseatci00@gmail.com"
!git config --global user.name "Ayseatci"
!git remote set-url origin https://{token}@github.com/Ayseatci/DI725_Assignment1.git

!git add {file_path}
!git commit -m "Updated preprocessed test data to be used for Roberta model"
!git push origin +dev

[dev 9446b35] Updated preprocessed test data to be used for Roberta model
 1 file changed, 1 insertion(+), 1 deletion(-)
Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 502 bytes | 502.00 KiB/s, done.
Total 5 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/Ayseatci/DI725_Assignment1.git
   d2078b3..9446b35  dev -> dev


The preprocessed test data is push to github repo.